# CA2, Phase 1: DCF and the slotted contention model

This notebook is **optional**: any language or tool is fine for the analysis (see the setup section of the assignment). It only **loads** simulation results; the model, the plots and the analysis are yours to write, in new cells under each section heading.

Rules:
- Your group's parameters come from `../params.json` (written by `ca2_params.py`). Do not type L or R by hand.
- Every number and plot in your report must be reproducible from the code you hand in; the TAs do not run it, but at the in-person delivery you will rerun parts of it and change them.

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from omnetpp.scave import results

with open(os.path.join("..", "params.json")) as f:
    P = json.load(f)
L, R = P["L"], P["R"]
print(f"students: {', '.join(P['students'])}")
print(f"L = {L} B, R = {R} Mbps")

## Loading results

`load_scalars(config)` returns every scalar of one config, with its iteration variables (for example `n` or `rtsThreshold`) and `repetition` as columns.
`scalar(sc, module, name)` returns one value per run: the scalar `name`, summed over all modules whose full path matches the regular expression `module`.

In [ ]:
RESULTS = "results"


def load_scalars(config):
    df = results.read_result_files(os.path.join(RESULTS, f"{config}-*.sca"))
    sc = results.get_scalars(df, include_itervars=True, include_runattrs=True)
    for col in ("n", "repetition"):
        if col in sc:
            sc[col] = pd.to_numeric(sc[col])
    return sc


def scalar(sc, module, name):
    rows = sc[(sc.name == name) & sc.module.str.fullmatch(module)]
    keys = [c for c in ("n", "rtsThreshold", "repetition") if c in rows and rows[c].notna().all()]
    return rows.groupby(keys).value.sum()


def load_vector(config, module, name):
    df = results.read_result_files(os.path.join(RESULTS, f"{config}-*.vec"))
    v = results.get_vectors(df)
    row = v[(v.module == module) & (v.name == name)].iloc[0]
    return row.vectime, row.vecvalue

Example: frames received by the AP in each run of the `Hidden` config (run that config first).

In [ ]:
hidden = load_scalars("Hidden")
scalar(hidden, r"Contention\.ap\.sink", "packets:count")

## 1-1  A single exchange up close

## 1-2  The slotted contention model

## 1-3  Model versus simulator

## 1-4  Hidden terminal and RTS/CTS